# 07. Calling a Function App MCP Server from a Foundry Model (Responses API)

**Difficulty: Advanced**

This notebook annotates `agent_with_functionapp_mcp.py`, in which a Foundry-deployed model (a gpt-5-mini-class deployment) answers *"What's the status of order ORD-1042?"* by calling this folder's C# Azure Function App — which exposes `GetOrderStatusTool` and `ListCustomerOrdersTool` over **MCP** — as a tool, via the **Responses API's built-in `mcp` tool type**.

It is the payoff of this sub-section: `connect_foundry_mcp.ipynb` registered the server centrally as a project connection; this script instead passes the server **inline** on a single API call — URL, auth header, and approval policy right in the `tools` list — and lets the *service* run the whole MCP round-trip (list tools → pick one → call it → fold the result into the answer) server-side, with no MCP client code in this file at all.

## Prerequisites

**pip3 packages** (all already in the repo root `requirements.txt`):
```bash
pip3 install azure-ai-projects azure-identity openai
```

**Azure resources required:**
- An Azure AI Foundry **project** with a Responses-API-capable model deployment (the course uses a gpt-5-mini-class deployment).
- The demo **Function App** in this folder deployed and reachable, with its MCP system key.

**Auth:** `az login` (`DefaultAzureCredential`); the Function App additionally wants its system key as an `x-functions-key` header, which the Responses API forwards on every MCP request.

**Env vars read via [`azure_config.py`](../../azure_config.py):** `AZURE_AI_PROJECT_ENDPOINT`, `AZURE_AI_MODEL_DEPLOYMENT`, `AZURE_FUNCTION_APP_HOST`, `MCP_SYSTEM_KEY`.

## What You'll Learn

- The Responses API's **hosted `mcp` tool type** — server URL + headers + `require_approval` declared inline, the whole tool round-trip executed by the service
- Why this script uses the Responses API at all: gpt-5-mini-class models **reject the `mcp` tool in the classic threads/runs Agent Service** (`azure-ai-agents`' `AgentsClient` fails with *"This model only supports Responses API compatible tools"*)
- How `AIProjectClient.get_openai_client()` bridges Foundry (Entra ID auth, project endpoint) to the standard `openai` SDK surface
- How to read the mixed `response.output` list — `mcp_list_tools`, `mcp_call`, and `message` items — to see the tool trace, and `response.output_text` for just the answer

### Step 1 — Repo-root config bootstrap and imports

The standard chapter bootstrap: walk up from the current directory to the folder holding [`azure_config.py`](../../azure_config.py) (no `__file__` in a notebook, so it starts at `Path.cwd()`), then read everything through the shared `config` object.

💡 **Exam tip:** `AIProjectClient` + `DefaultAzureCredential` is the canonical *keyless* entry point to a Foundry project — the exam consistently prefers Entra ID (`az login`, managed identities) over API keys for Azure-to-Azure calls. The only raw key in this file is the Function App's own `x-functions-key`, which belongs to the Functions webhook, not to Foundry.

In [ ]:
import sys
from pathlib import Path

_start = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
for _parent in [_start, *_start.parents]:
    if (_parent / "azure_config.py").exists():
        sys.path.insert(0, str(_parent))
        break

from azure_config import config

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

PROJECT_ENDPOINT = config.project_endpoint
DEPLOYMENT = config.model_deployment

FUNCTION_APP_HOST = config.function_app_host
MCP_SYSTEM_KEY = config.mcp_system_key

### Step 2 — The inline `mcp` tool definition

One dict describes the entire remote server to the Responses API:

- `type: "mcp"` — a *hosted* tool: Azure's service (not your Python process) connects to the server, lists its tools, and calls them mid-response.
- `server_label` — a short handle; it shows up in the response's `mcp_list_tools` / `mcp_call` output items so traces stay readable.
- `server_url` — the Functions MCP **SSE** webhook, `https://<host>/runtime/webhooks/mcp/sse`, the same endpoint `connect_foundry_mcp.ipynb` stored as a connection `target`.
- `headers` — forwarded verbatim on every request to the server; here the `x-functions-key` system key that guards the Functions MCP extension.
- `require_approval: "never"` — tool calls execute without a human-in-the-loop pause. The default requires approvals, which would surface `mcp_approval_request` items you'd have to answer with a follow-up call.

💡 **Exam tip:** distinguish the **three MCP integration styles** in this course: (1) *client-side* — your own Python MCP client calls tools and feeds results to the model (`10_mcp/` chapter); (2) *hosted via Responses API* — this inline dict; (3) *hosted via a project connection* — the previous notebook, where Foundry stores URL + key centrally. Same server, three wiring choices.

🔄 **Alternatives:** `require_approval: "always"` (or a per-tool allow-list dict) for human-in-the-loop; an `allowed_tools` list to expose only a subset of the server's tools; for models that support it, `azure-ai-agents`' `McpTool` in the classic Agent Service — which is exactly what gpt-5-mini refuses, per the header comment in the script.

In [ ]:
mcp_tool = {
    "type": "mcp",
    "server_label": "soubhik_demo_funcapp_mcp",
    "server_url": f"https://{FUNCTION_APP_HOST}/runtime/webhooks/mcp/sse",
    "headers": {"x-functions-key": MCP_SYSTEM_KEY},
    "require_approval": "never",
}

### Step 3 — One Responses API call, tool round-trip included

`AIProjectClient(...).get_openai_client()` returns a standard `openai`-SDK `AzureOpenAI` client already pointed at the project's OpenAI-compatible endpoint and authenticated with your Entra ID token — no key, no endpoint string beyond the project's.

Then a single `client.responses.create(...)` does everything: the service sends the prompt + tool schema to the model, the model decides to call `get_order_status`, the *service* performs the MCP call to the Function App (forwarding the `x-functions-key` header), and the model composes the final answer — all before the call returns.

The loop over `response.output` prints each output item's `type` to make that visible: expect an `mcp_list_tools` item (the tool discovery), one or more `mcp_call` items (each named via `item.name`), then a `message` item. `response.output_text` is the SDK's shortcut that concatenates just the message text.

🔄 **Alternatives:** `stream=True` would yield these as typed events while they happen; a plain `AzureOpenAI(api_key=..., azure_endpoint=...)` client against the same deployment works too but reintroduces key management — `get_openai_client()` exists precisely to avoid that.

In [ ]:
def main():
    project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential())
    client = project_client.get_openai_client()

    response = client.responses.create(
        model=DEPLOYMENT,
        input="What's the status of order ORD-1042?",
        tools=[mcp_tool],
    )

    for item in response.output:
        print(f"[{item.type}]", getattr(item, "name", "") or "")

    print("\nFinal answer:")
    print(response.output_text)


main()

## Summary

This notebook annotated `agent_with_functionapp_mcp.py`: a Foundry model calling this folder's Function App MCP server through the Responses API's hosted `mcp` tool — server URL, `x-functions-key` header, and `require_approval: "never"` all declared inline on one `responses.create` call, with the service executing the MCP round-trip and the `response.output` items (`mcp_list_tools` → `mcp_call` → `message`) exposing the trace. The framing detail worth remembering: gpt-5-mini-class models only accept MCP through the Responses API — the classic threads/runs Agent Service rejects the `mcp` tool type for them — which is why this file uses `get_openai_client()` rather than `azure-ai-agents`.

## Try It Yourself

1. **Easy:** Change the `input` to *"List all orders for customer CUST-7"* and confirm the model switches to the `ListCustomerOrdersTool` — visible in the printed `mcp_call` item name.
2. **Intermediate:** Set `require_approval` to `"always"`, observe the `mcp_approval_request` item in `response.output`, and answer it with a second `responses.create` call carrying an `mcp_approval_response` input item.
3. **Advanced:** Ask a question needing *both* tools ("Find CUST-7's most recent order and tell me its status"), print the full item trace, and compare this hosted round-trip with the client-side agentic loop you built by hand in `10_mcp/03_mcp_client/` — same protocol, opposite side of the wire.